# Convertir el modelo de Tuki ya entrenado al formato del teléfono (.task) — Kaggle

**Antes de ejecutar**
1. *Add Input → Your Work → Notebooks* → agrega la salida del cuaderno de entrenamiento (la versión que
   terminó). Debe contener la carpeta `gemma_aikukisna_v3/`.
2. *Add-ons → Secrets*: activa `HF_TOKEN` para este cuaderno (se usa para bajar `tokenizer.model`).
3. *Session options*: **Accelerator: None** (no hace falta GPU) e **Internet: On**.

**Ejecutar:** *Save Version → Save & Run All (Commit)*. Tarda unos 30–40 min. El `.task` queda en **Output**.

In [ ]:
!pip -q install litert-torch mediapipe
# litert-lm-builder (opcional, solo para el formato .litertlm) trae código protobuf 6, pero mediapipe
# deja instalado protobuf 5 y la importación falla con VersionError. litert-torch lo importa dentro de
# un try/except ImportError, así que basta con quitarlo para que convierta a .tflite normalmente.
!pip -q uninstall -y litert-lm-builder litert_lm_builder
import importlib.util, os, shutil
_spec = importlib.util.find_spec("litert_lm_builder")
if _spec and _spec.submodule_search_locations:
    shutil.rmtree(list(_spec.submodule_search_locations)[0])
importlib.invalidate_caches()
assert importlib.util.find_spec("litert_lm_builder") is None, "litert_lm_builder sigue instalado"

In [ ]:
import glob, os
from huggingface_hub import HfApi, login
from huggingface_hub.errors import GatedRepoError
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN").strip()
login(HF_TOKEN)
print("Hugging Face:", HfApi(token=HF_TOKEN).whoami()["name"])
# Falla aquí, con un mensaje claro, si el token no puede leer Gemma (licencia sin aceptar o token
# "fine-grained" sin permiso para repos restringidos).
try:
    HfApi(token=HF_TOKEN).auth_check("google/gemma-3-1b-it")
except GatedRepoError as e:
    raise SystemExit(
        "El token no tiene acceso a google/gemma-3-1b-it: acepta la licencia en "
        "https://huggingface.co/google/gemma-3-1b-it con la misma cuenta y usa un token de tipo Read"
    ) from e

configs = glob.glob("/kaggle/input/**/gemma_aikukisna_v3/config.json", recursive=True)
assert configs, "No encuentro gemma_aikukisna_v3/: agrega la salida del cuaderno de entrenamiento con Add Input"
MODELO = os.path.dirname(configs[0])
print(MODELO, os.listdir(MODELO))

In [ ]:
from huggingface_hub import hf_hub_download
from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.layers import kv_cache
from litert_torch.generative.utilities import converter
from litert_torch.generative.utilities.export_config import ExportConfig

# save_pretrained solo guarda tokenizer.json; MediaPipe necesita el tokenizer.model de SentencePiece,
# que es el mismo del modelo base (el ajuste no cambia el vocabulario).
TOKENIZADOR = hf_hub_download("google/gemma-3-1b-it", "tokenizer.model", token=globals().get("HF_TOKEN"))

KV_MAX = 1536
pytorch_model = gemma3.build_model_1b(MODELO)
config = ExportConfig()
config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
config.mask_as_input = True
converter.convert_to_tflite(
    pytorch_model,
    output_path="salida_tflite",
    output_name_prefix="gemma3_aikukisna",
    prefill_seq_len=[128, 512],
    kv_cache_max_len=KV_MAX,
    quantize="dynamic_int4_block32",
    export_config=config,
)
!ls -la salida_tflite

In [ ]:
# El mediapipe que se instala en Python 3.13 ya no trae el empaquetador (mediapipe.tasks.python.genai).
# Se crea un Python 3.12 aparte con uv solo para este paso, con una versión que sí lo incluye.
!pip -q install uv
!uv venv -q -p 3.12 /tmp/mp312
!uv pip install -q --python /tmp/mp312 "mediapipe==0.10.18" sentencepiece
import glob, os, subprocess

tflite = glob.glob("salida_tflite/*.tflite")[0]
script = f'''
from mediapipe.tasks.python.genai import bundler
bundler.create_bundle(bundler.BundleConfig(
    tflite_model={tflite!r},
    tokenizer_model={TOKENIZADOR!r},
    start_token="<bos>",
    stop_tokens=["<eos>", "<end_of_turn>"],
    output_filename="gemma3-1b-aikukisna-v3-int4.task",
))
'''
# Sin prompt_prefix/suffix: GemmaTraductorLocal y GemmaAssistantEngine ya arman los turnos.
# MPLBACKEND=Agg: el cuaderno define un backend de matplotlib que el Python aparte no conoce.
subprocess.run(["/tmp/mp312/bin/python", "-c", script], check=True, env={**os.environ, "MPLBACKEND": "Agg"})
!ls -la gemma3-1b-aikukisna-v3-int4.task

!rm -rf salida_tflite  # solo se conserva el .task en Output

## Descargar

Pestaña **Output** → `gemma3-1b-aikukisna-v3-int4.task` (unos 600–700 MB). Se publica en GitHub como
Release `modelo-tuki-v3`; después, en `local.properties` de la app: `TUKI_MODELO_AJUSTADO=true`.